# CT-CBM Noise-Control Colab Notebook

This notebook runs a noise-control version of CT-CBM. It intentionally skips concept annotation, CAV discovery, LIG ranking, and concept ranking from real concepts.

The goal is to keep the downstream CT-CBM pipeline identical while replacing the concept bank with random binary noise concepts saved under the same artifact names expected by the original code.

Pipeline:
1. Load a text-classification dataset and fine-tune the black-box encoder.
2. Generate random binary `dummy_<concept>` columns for train/test and save them like real `ours` annotations.
3. Save fake CAV/ranking/detection artifacts with the same filenames as the real pipeline.
4. Run the original CT-CBM iterative training code on the noise concepts.
5. Print test classification reports for CT-CBM simple and CT-CBM residual branches.

All explanatory comments inside code cells are written in English.

In [ ]:
# Colab setup.
# Run this once in a fresh Colab runtime.

!pip -q install   transformers datasets accelerate sentence-transformers   scikit-learn pandas numpy tqdm matplotlib tabulate joblib   umap-learn hdbscan captum

In [ ]:
# Repository setup.
# Option A: set REPO_URL and let Colab clone the project.
# Option B: upload/mount the repository and set PROJECT_DIR manually.

import os
from pathlib import Path

REPO_URL = "https://github.com/dauduchieu/iSE-CT-CBM.git"
PROJECT_DIR = Path("/content/CT-CBM")

if not PROJECT_DIR.exists() and REPO_URL:
    !git clone {REPO_URL} {PROJECT_DIR}

if not PROJECT_DIR.exists() and Path.cwd().name == "CT-CBM":
    PROJECT_DIR = Path.cwd()

if not (PROJECT_DIR / "run_experiments").exists():
    raise FileNotFoundError("Cannot find run_experiments/. Set REPO_URL or PROJECT_DIR correctly.")

os.chdir(PROJECT_DIR)
print(f"Project directory: {PROJECT_DIR}")

In [ ]:
# Imports and path setup.

import sys, gc, json, pickle, random, warnings, logging
from pathlib import Path

ROOT = Path.cwd()
for rel in ["run_experiments", "run_experiments/scripts", "run_experiments/models", "run_experiments/data"]:
    path = str(ROOT / rel)
    if path not in sys.path:
        sys.path.append(path)

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")
for logger in [logging.getLogger(name) for name in logging.root.manager.loggerDict]:
    if "transformers" in logger.name.lower():
        logger.setLevel(logging.ERROR)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

In [ ]:
# Experiment configuration.
# Change DATASET to any dataset supported by run_experiments/scripts/load_config.py.

MODEL_NAME = "bert-base-uncased"
DATASET = "agnews"  # Options include: agnews, movies, dbpedia, medical, pubmed, stackoverflow, medical_2, ecom, sst2.
ANNOTATION = "our_annotation"  # Keep this name so downstream filenames match the original CT-CBM code.

RUN_BLACKBOX_TRAINING = True
RUN_NOISE_ARTIFACTS = True
RUN_CT_CBM_TRAINING = True

# Debug controls. Use None for full data.
MAX_TRAIN_ROWS = 400
MAX_VAL_ROWS = 200
MAX_TEST_ROWS = 200

# Black-box fitting controls.
BLACKBOX_NUM_EPOCHS = 5

# Noise concept controls.
NOISE_CONCEPT_COUNT = 100  # None means max(32, 3 * num_labels). Must be >= 3 * num_labels for CT-CBM initialization.
NOISE_POSITIVE_RATE = 0.30  # Bernoulli probability for each random concept label.
NOISE_CONCEPT_NAME_LENGTH = 12  # Length of each deterministic random concept string.

# CT-CBM fitting controls.
NUM_ITERATIONS = 5
NUM_EPOCHS_JOINT = 5
NUM_EPOCHS_RESIDUAL = 5
COVERAGE_THRESHOLD = 0.80

In [ ]:
# Load config and redirect outputs to Colab storage.

from load_config import load_config

config = load_config(MODEL_NAME, DATASET)
config.annotation = ANNOTATION
config.DATASET = DATASET
config.model_name = MODEL_NAME
config.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config.num_epochs = NUM_EPOCHS_JOINT
config.cavs_type = "mean"
config.cavs_type_arg = "mean"
config.agg_mode = "abs"
config.agg_scope = "all"
config.seed = SEED

OUTPUT_ROOT = Path("/content/ct_cbm_outputs_noise") / DATASET
config.SAVE_PATH = str(OUTPUT_ROOT) + "/"
config.SAVE_PATH_CONCEPTS = str(OUTPUT_ROOT / "concepts_discovery")
Path(config.SAVE_PATH_CONCEPTS).mkdir(parents=True, exist_ok=True)
(Path(config.SAVE_PATH) / "blue_checkpoints" / config.model_name / "cavs" / config.cavs_type).mkdir(parents=True, exist_ok=True)
(Path(config.SAVE_PATH) / "blue_checkpoints" / config.model_name / "Our_CBM_joint").mkdir(parents=True, exist_ok=True)

print("SAVE_PATH:", config.SAVE_PATH)
print("SAVE_PATH_CONCEPTS:", config.SAVE_PATH_CONCEPTS)

In [ ]:
# Load task data.

from prepare_data import load_fc_prepare_data

prepare_data = load_fc_prepare_data(config.DATASET)
train_loader, test_loader, val_loader, train_df, val_df, test_df = prepare_data(config)

if MAX_TRAIN_ROWS is not None:
    train_df = train_df.head(MAX_TRAIN_ROWS).copy()
if MAX_VAL_ROWS is not None:
    val_df = val_df.head(MAX_VAL_ROWS).copy()
if MAX_TEST_ROWS is not None:
    test_df = test_df.head(MAX_TEST_ROWS).copy()

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    assert {"text", "label"}.issubset(df.columns), f"{name}_df must contain text and label"
    df["text"] = df["text"].astype(str).str.strip()
    df["label"] = df["label"].astype(int)
    print(name, df.shape, df.head(1).to_dict("records"))

In [ ]:
# Load the encoder/tokenizer and a classifier head for black-box fine-tuning.

from models.utils import load_model_and_tokenizer, RidgeLinearLayer

embedder_model, embedder_tokenizer, ModelXtoCtoY_layer, blackbox_classifier = load_model_and_tokenizer(config, n_concepts=1)
try:
    getattr(embedder_tokenizer, "encode_plus")
except AttributeError:
    embedder_tokenizer.encode_plus = embedder_tokenizer.__call__
embedder_model.to(config.device)
print("Loaded embedder:", MODEL_NAME)

# Black-box BERT Fine-Tuning

The noise-control still fine-tunes the black-box encoder first. CT-CBM then reuses this fine-tuned encoder, but receives random noise concepts instead of discovered macro-concepts.

In [ ]:
# Fine-tune the black-box classifier on task labels.

from concepts_bank_utils import create_dataloader

blackbox_train_loader = create_dataloader(train_df[["text", "label"]].copy(), embedder_tokenizer, config.max_len, config.batch_size, shuffle=True)
blackbox_val_loader = create_dataloader(val_df[["text", "label"]].copy(), embedder_tokenizer, config.max_len, config.batch_size, shuffle=False)
blackbox_test_loader = create_dataloader(test_df[["text", "label"]].copy(), embedder_tokenizer, config.max_len, config.batch_size, shuffle=False)

_original_num_epochs = config.num_epochs
config.num_epochs = BLACKBOX_NUM_EPOCHS

if config.model_name == "gemma":
    from models.BaselineModel_gemma import BaselineModel
else:
    from models.BaselineModel import BaselineModel

black_box_model = BaselineModel(
    embedder_model=embedder_model,
    classifier=blackbox_classifier,
    train_loader=blackbox_train_loader,
    val_loader=blackbox_val_loader,
    test_loader=blackbox_test_loader,
    config=config,
    save_path=config.SAVE_PATH,
    use_cls_token=config.use_cls_token,
)

if RUN_BLACKBOX_TRAINING:
    black_box_model.train_model()
    black_box_model.load_model()
    black_box_model.evaluate_model(blackbox_test_loader, "Test")
    black_box_model.save_performance_json()
else:
    black_box_model.load_model()
    print("Loaded existing fine-tuned black-box checkpoint.")

config.num_epochs = _original_num_epochs

# Reuse the fine-tuned black-box encoder for CT-CBM.
embedder_model = black_box_model.embedder_model
_tmp_encoder, embedder_tokenizer, ModelXtoCtoY_layer, _ = load_model_and_tokenizer(config, n_concepts=1)
del _tmp_encoder
try:
    getattr(embedder_tokenizer, "encode_plus")
except AttributeError:
    embedder_tokenizer.encode_plus = embedder_tokenizer.__call__
embedder_model.to(config.device)
ModelXtoCtoY_layer.to(config.device)
torch.cuda.empty_cache()
print("Loaded the fine-tuned black-box encoder for noise CT-CBM.")

# Noise Concept Artifacts

This section replaces concept annotation and ranking. It creates random binary concept columns and saves them under the same filenames that the original CT-CBM pipeline expects from real `ours` annotation/ranking.

In [ ]:
# Create random binary noise concepts and save them like real annotation artifacts.

from concepts_bank_utils import clean_concept_name
import string

rng = np.random.default_rng(SEED)
noise_count = NOISE_CONCEPT_COUNT or max(32, 3 * int(config.num_labels))
noise_count = max(noise_count, 3 * int(config.num_labels))


def generate_random_concept_names(count, length, rng):
    # Generate deterministic random strings that are valid DataFrame/DataLoader keys.
    # The first character is always a letter so each concept remains a clean identifier-like name.
    alphabet_first = np.array(list(string.ascii_lowercase))
    alphabet_rest = np.array(list(string.ascii_lowercase + string.digits))
    names = set()
    while len(names) < count:
        first = rng.choice(alphabet_first)
        rest = "".join(rng.choice(alphabet_rest, size=length - 1).tolist())
        names.add(str(first) + rest)
    return sorted(names)


noise_concepts = generate_random_concept_names(noise_count, NOISE_CONCEPT_NAME_LENGTH, rng)


def add_noise_concepts(df, concepts, positive_rate, rng):
    out = df[["text", "label"]].copy().reset_index(drop=True)
    for concept in concepts:
        out[f"dummy_{concept}"] = rng.binomial(1, positive_rate, size=len(out)).astype(int)
    return out

if RUN_NOISE_ARTIFACTS:
    df_aug_train = add_noise_concepts(train_df, noise_concepts, NOISE_POSITIVE_RATE, rng)
    df_aug_test = add_noise_concepts(test_df, noise_concepts, NOISE_POSITIVE_RATE, rng)

    concepts_dir = Path(config.SAVE_PATH_CONCEPTS)
    df_aug_train.to_csv(concepts_dir / "df_with_topics_v4.csv", index=False)
    df_aug_test.to_csv(concepts_dir / "df_with_topics_v4_test.csv", index=False)

    # Minimal placeholder files help humans inspect the run and mirror real annotation outputs.
    pd.DataFrame({"concept": noise_concepts, "cluster": range(len(noise_concepts))}).to_csv(concepts_dir / "df_cluster.csv", index=False)
    with open(concepts_dir / "number_to_macro.json", "w") as f:
        json.dump({str(i): c for i, c in enumerate(noise_concepts)}, f, indent=2)
    with open(concepts_dir / "concept_to_macro.json", "w") as f:
        json.dump({c: c for c in noise_concepts}, f, indent=2)
else:
    df_aug_train = pd.read_csv(Path(config.SAVE_PATH_CONCEPTS) / "df_with_topics_v4.csv")
    df_aug_test = pd.read_csv(Path(config.SAVE_PATH_CONCEPTS) / "df_with_topics_v4_test.csv")
    noise_concepts = [col.replace("dummy_", "") for col in df_aug_train.columns if col.startswith("dummy_")]

print("Noise concepts:", len(noise_concepts))
print("Augmented train:", df_aug_train.shape)
print("Augmented test:", df_aug_test.shape)

In [ ]:
# Build dataloaders from noise-augmented data.

train_aug_loader = create_dataloader(df_aug_train, embedder_tokenizer, config.max_len, config.batch_size, shuffle=True)
test_aug_loader = create_dataloader(df_aug_test, embedder_tokenizer, config.max_len, config.batch_size, shuffle=False)
val_loader = create_dataloader(val_df[["text", "label"]].copy(), embedder_tokenizer, config.max_len, config.batch_size, shuffle=False)

print("Train batches:", len(train_aug_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_aug_loader))

In [ ]:
# Save fake CAV/detection/ranking artifacts with the same names as real concept artifacts.
# The downstream CT-CBM training only needs the coverage ranking JSON, but saving
# the surrounding files makes this noise run easier to compare with normal runs.

ranking_dir = Path(config.SAVE_PATH) / "blue_checkpoints" / config.model_name / "cavs" / config.cavs_type
ranking_dir.mkdir(parents=True, exist_ok=True)

# Fake random CAVs are saved for artifact completeness.
fake_cavs = {concept: rng.normal(0, 1, size=int(config.dim)).astype(float).tolist() for concept in noise_concepts}
with open(ranking_dir / f"cavs_mean_{config.annotation}.json", "w") as f:
    json.dump(fake_cavs, f)

# Fake concept identifiability metrics. These are not used for learning labels,
# but mimic the detection_concept file written by the real ranking pipeline.
detection_metrics = {}
for concept in noise_concepts:
    positive_rate = float(df_aug_train[f"dummy_{concept}"].mean())
    detection_metrics[concept] = {
        "accuracy": 0.5,
        "positive_rate": positive_rate,
        "F1": 0.5,
        "TP": 0,
        "FP": 0,
        "FN": 0,
        "TN": 0,
    }
with open(ranking_dir / f"detection_concept_{config.cavs_type}_{config.annotation}_{config.agg_mode}_{config.agg_scope}.json", "w") as f:
    json.dump(detection_metrics, f, indent=2)

# Rank noise concepts by deterministic random scores.
noise_scores = {concept: float(score) for concept, score in zip(noise_concepts, rng.random(len(noise_concepts)))}
sorted_concepts = sorted(noise_scores.items(), key=lambda item: item[1], reverse=True)
with open(ranking_dir / "sorted_macro_concepts.json", "w") as f:
    json.dump(dict(sorted_concepts), f, indent=2)
with open(ranking_dir / f"combined_score_LIG_{config.annotation}_{config.agg_mode}_{config.agg_scope}.json", "w") as f:
    json.dump(dict(sorted_concepts), f, indent=2)

# Compute cumulative coverage exactly like the normal notebook expects.
def compute_noise_coverage(df, sorted_concepts):
    total = max(len(df), 1)
    cumulative_cols = []
    coverage = {}
    for concept, _score in sorted_concepts:
        col = f"dummy_{concept}"
        if col not in df.columns:
            continue
        cumulative_cols.append(col)
        covered = df[cumulative_cols].eq(1).any(axis=1).sum()
        coverage[concept] = float(covered / total)
    return coverage

coverage_ranking = compute_noise_coverage(df_aug_train, sorted_concepts)
coverage_path = ranking_dir / f"sorted_macro_concepts_coverage_{config.annotation}_{config.agg_mode}_{config.agg_scope}.json"
with open(coverage_path, "w") as f:
    json.dump(coverage_ranking, f, indent=2)

print("Noise coverage file:", coverage_path)
print("Top noise coverage entries:", list(coverage_ranking.items())[:10])

In [ ]:
# Initialize the original CT-CBM pipeline on the fine-tuned encoder and noise concepts.

import importlib

if config.model_name == "gemma":
    import full_pipeline_gemma as _full_pipeline_module
    import models.jointCBMv2_gemma as _joint_module
else:
    import full_pipeline as _full_pipeline_module
    import models.jointCBMv2 as _joint_module

_full_pipeline_module = importlib.reload(_full_pipeline_module)
_joint_module = importlib.reload(_joint_module)
JointResidualFittingModel = _full_pipeline_module.JointResidualFittingModel
CTJointModel = _joint_module.JointModel

ct_joint = CTJointModel(embedder_model, embedder_tokenizer, ModelXtoCtoY_layer, config, train_aug_loader, val_loader)

linear_layer = RidgeLinearLayer(config.dim, config.num_labels, l2_lambda=config.l2_lambda)
linear_layer.to(config.device)
ct_joint.linear_layer = linear_layer

joint_residual_model = JointResidualFittingModel(
    joint_model=ct_joint,
    linear_layer=linear_layer,
    discovery_model=None,
    discovery_tokenizer=None,
    config=config,
)

print("Noise CT-CBM initialized.")

In [ ]:
# Run CT-CBM on the noise concept bank.

if RUN_CT_CBM_TRAINING:
    joint_residual_model.run_full_pipeline_tcavs_strategy(
        train_loader=train_aug_loader,
        val_loader=val_loader,
        test_loader=test_aug_loader,
        num_iterations=NUM_ITERATIONS,
        num_epochs_residual_layer=NUM_EPOCHS_RESIDUAL,
        cavs_type_arg="mean",
        strategy="new_heuristique",
        coverage_threshold=COVERAGE_THRESHOLD,
    )
else:
    print("Skipping CT-CBM training.")

In [ ]:
# CT-CBM noise-control test classification reports.

from sklearn.metrics import classification_report

if "joint_residual_model" not in globals():
    raise RuntimeError("Run the CT-CBM initialization/training cells before this report cell.")

ct_cbm_model = joint_residual_model.joint_model
ct_cbm_model.embedder_model.eval()
ct_cbm_model.ModelXtoCtoY_layer.eval()
if getattr(ct_cbm_model, "linear_layer", None) is not None:
    ct_cbm_model.linear_layer.eval()

y_true = []
y_pred_simple = []
y_pred_residual = []

with torch.no_grad():
    for batch in test_aug_loader:
        labels = batch["label"].to(config.device)

        outputs, _ = ct_cbm_model.forward(batch)
        simple_logits = outputs[0]
        simple_preds = torch.argmax(simple_logits, dim=1)

        input_ids = batch["input_ids"].to(config.device)
        attention_mask = batch["attention_mask"].to(config.device)
        residual_logits, _, _ = ct_cbm_model.forward_linear(input_ids, attention_mask)
        residual_preds = torch.argmax(residual_logits, dim=1)

        y_true.extend(labels.cpu().tolist())
        y_pred_simple.extend(simple_preds.cpu().tolist())
        y_pred_residual.extend(residual_preds.cpu().tolist())

print("Noise CT-CBM simple classification report")
print(classification_report(y_true, y_pred_simple, digits=4, zero_division=0))

print("Noise CT-CBM residual classification report")
print(classification_report(y_true, y_pred_residual, digits=4, zero_division=0))

noise_report_df = pd.DataFrame({
    "label": y_true,
    "noise_ct_cbm_simple_pred": y_pred_simple,
    "noise_ct_cbm_residual_pred": y_pred_residual,
})
report_path = Path(config.SAVE_PATH) / "noise_ct_cbm_test_predictions_simple_and_residual.csv"
noise_report_df.to_csv(report_path, index=False)
print("Saved noise CT-CBM test predictions to", report_path)

In [ ]:
# Inspect saved noise-control outputs.

for path in sorted(Path(config.SAVE_PATH).rglob("*.json")):
    print(path)
for path in sorted(Path(config.SAVE_PATH).rglob("*.pth")):
    print(path)